In [0]:
fligths = spark.read.table('flights')
bookings = spark.read.table('bookings')
preferences = spark.read.table('preferences')

Part 3 : Transformations

In [0]:
from pyspark.sql.functions import *
bookings.withColumn(
    "revenue",
    col("ticket_price")
).withColumn(
    "price_band",
    when(col("ticket_price") > 20000, "Premium")
    .when(col("ticket_price") > 10000, "Standard")
    .otherwise("Budget")
).show()

+----------+---------+--------------+---------------+------------+------------+-------+----------+
|booking_id|flight_id|passenger_name|   travel_class|ticket_price|booking_date|revenue|price_band|
+----------+---------+--------------+---------------+------------+------------+-------+----------+
|     B1001|     F101|  Rahul Sharma|        Economy|        8500|  2026-06-01|   8500|    Budget|
|     B1002|     F101|   Priya Reddy|       Business|       22000|  2026-06-01|  22000|   Premium|
|     B1003|     F102|    Amit Kumar|        Economy|        9000|  2026-06-02|   9000|    Budget|
|     B1004|     F103|   Sneha Patel|Premium Economy|       15000|  2026-06-02|  15000|  Standard|
|     B1005|     F104|    Farhan Ali|        Economy|        7500|  2026-06-03|   7500|    Budget|
|     B1006|     F105|    Neha Singh|       Business|       25000|  2026-06-03|  25000|   Premium|
|     B1007|     F106|   Arjun Verma|        Economy|       10000|  2026-06-04|  10000|    Budget|
|     B100

In [0]:
fligths.withColumn(
    'delay_flag',
    when(col('status') == 'Delayed', 'Yes')
    .otherwise('No')
).show()

+---------+---------+---------+---------+--------+---------+----------+
|flight_id|  airline|from_city|  to_city|duration|   status|delay_flag|
+---------+---------+---------+---------+--------+---------+----------+
|     F101|   Indigo|Hyderabad|    Delhi|     140|  On Time|        No|
|     F102|Air India|   Mumbai|  Chennai|     120|  Delayed|       Yes|
|     F103|  Vistara|Bangalore|Hyderabad|      90|  On Time|        No|
|     F104|   Indigo|    Delhi|   Mumbai|     130|Cancelled|        No|
|     F105|Air India|  Chennai|Bangalore|      80|  On Time|        No|
|     F106|    Akasa|     Pune|    Delhi|     150|  Delayed|       Yes|
|     F107|  Vistara|Hyderabad|  Kolkata|     160|  On Time|        No|
|     F108|   Indigo|   Mumbai|Hyderabad|     110|  On Time|        No|
|     F109|    Akasa|    Delhi|  Chennai|     145|  Delayed|       Yes|
|     F110|Air India|Bangalore|   Mumbai|      95|  On Time|        No|
|     F111|   Indigo|Hyderabad|      Goa|      75|  On Time|    

In [0]:
passenger_journey = (
    bookings
    .join(fligths , "flight_id", "left")
    .join(preferences, "passenger_name", "left")
)
passenger_journey.show()

+--------------+---------+----------+---------------+------------+------------+---------+---------+---------+--------+---------+-------------+-------+------+
|passenger_name|flight_id|booking_id|   travel_class|ticket_price|booking_date|  airline|from_city|  to_city|duration|   status|extra_baggage|   meal|  seat|
+--------------+---------+----------+---------------+------------+------------+---------+---------+---------+--------+---------+-------------+-------+------+
|  Rahul Sharma|     F101|     B1001|        Economy|        8500|  2026-06-01|   Indigo|Hyderabad|    Delhi|     140|  On Time|         true|    Veg|Window|
|   Priya Reddy|     F101|     B1002|       Business|       22000|  2026-06-01|   Indigo|Hyderabad|    Delhi|     140|  On Time|        false|Non-Veg| Aisle|
|    Amit Kumar|     F102|     B1003|        Economy|        9000|  2026-06-02|Air India|   Mumbai|  Chennai|     120|  Delayed|        false|    Veg|Middle|
|   Sneha Patel|     F103|     B1004|Premium Economy

In [0]:
fligths.createOrReplaceTempView('fligths')
bookings.createOrReplaceTempView('bookings')
preferences.createOrReplaceTempView('preferences')

In [0]:
%sql
select airline, sum(ticket_price) as revenue 
from bookings
join flights on bookings.flight_id = flights.flight_id
group by airline;

airline,revenue
Air India,68000
Akasa,62000
Indigo,90000
Vistara,71500


In [0]:
%sql
select from_city, to_city, sum(ticket_price) as revenue 
from bookings
join flights on bookings.flight_id = flights.flight_id
group by from_city, to_city;

from_city,to_city,revenue
Delhi,Chennai,28000
Hyderabad,Delhi,39000
Goa,Delhi,7000
Delhi,Mumbai,7500
Mumbai,Chennai,9000
Pune,Delhi,10000
Bangalore,Mumbai,23500
Delhi,Hyderabad,18000
Kolkata,Bangalore,10500
Chennai,Bangalore,25000


In [0]:
%sql
select avg(ticket_price) as avg_price 
from bookings;

avg_price
14575.0


In [0]:
%sql
select to_city as most_popular_destination 
from bookings
join flights on bookings.flight_id = flights.flight_id
group by to_city
order by count(*) desc
limit 1;

most_popular_destination
Delhi


Part 6 : Window Functions

In [0]:
from pyspark.sql.window import Window

flight_revenue = passenger_journey.groupBy(
    "flight_id"
).agg(sum("ticket_price").alias("revenue"))

win_spec = Window.orderBy(col("revenue").desc())

flight_revenue.withColumn(
    "rank",
    row_number().over(win_spec)
).filter(col("rank") <= 3).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------+----+
|flight_id|revenue|rank|
+---------+-------+----+
|     F101|  39000|   1|
|     F103|  38000|   2|
|     F109|  28000|   3|
+---------+-------+----+



In [0]:
route_df = passenger_journey.groupBy(
    "airline",
    "from_city",
    "to_city"
).agg(count("*").alias("route_count"))
display(route_df)

airline,from_city,to_city,route_count
Indigo,Delhi,Hyderabad,1
Akasa,Delhi,Chennai,1
Indigo,Hyderabad,Goa,1
Indigo,Hyderabad,Delhi,3
Indigo,Delhi,Mumbai,1
Vistara,Hyderabad,Kolkata,2
Vistara,Bangalore,Hyderabad,2
Akasa,Pune,Delhi,1
Indigo,Mumbai,Hyderabad,1
Air India,Kolkata,Bangalore,1


In [0]:
win_spec = Window.partitionBy("airline").orderBy(col("route_count").desc())

route_df.withColumn(
    "rank",
    rank().over(win_spec)
).filter(col("rank") == 1).show()

+---------+---------+---------+-----------+----+
|  airline|from_city|  to_city|route_count|rank|
+---------+---------+---------+-----------+----+
|Air India|Bangalore|   Mumbai|          2|   1|
|    Akasa|    Delhi|  Chennai|          1|   1|
|    Akasa|     Pune|    Delhi|          1|   1|
|    Akasa|  Chennai|     Pune|          1|   1|
|   Indigo|Hyderabad|    Delhi|          3|   1|
|  Vistara|Hyderabad|  Kolkata|          2|   1|
|  Vistara|Bangalore|Hyderabad|          2|   1|
+---------+---------+---------+-----------+----+



In [0]:
win_spec = Window.orderBy(
    "booking_id"
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

bookings.withColumn(
    "running_revenue",
    sum("ticket_price").over(win_spec)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+---------+--------------+---------------+------------+------------+---------------+
|booking_id|flight_id|passenger_name|   travel_class|ticket_price|booking_date|running_revenue|
+----------+---------+--------------+---------------+------------+------------+---------------+
|     B1001|     F101|  Rahul Sharma|        Economy|        8500|  2026-06-01|           8500|
|     B1002|     F101|   Priya Reddy|       Business|       22000|  2026-06-01|          30500|
|     B1003|     F102|    Amit Kumar|        Economy|        9000|  2026-06-02|          39500|
|     B1004|     F103|   Sneha Patel|Premium Economy|       15000|  2026-06-02|          54500|
|     B1005|     F104|    Farhan Ali|        Economy|        7500|  2026-06-03|          62000|
|     B1006|     F105|    Neha Singh|       Business|       25000|  2026-06-03|          87000|
|     B1007|     F106|   Arjun Verma|        Economy|       10000|  2026-06-04|          97000|
|     B1008|     F107|    Meera Nair|Pre

In [0]:
airline_revenue = passenger_journey.groupBy(
    "airline"
).agg(sum("ticket_price").alias("revenue"))

win_spec = Window.orderBy(col("revenue").desc())

airline_revenue.withColumn(
    "rank",
    rank().over(win_spec)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------+----+
|  airline|revenue|rank|
+---------+-------+----+
|   Indigo|  90000|   1|
|  Vistara|  71500|   2|
|Air India|  68000|   3|
|    Akasa|  62000|   4|
+---------+-------+----+



In [0]:
destination_df = passenger_journey.groupBy(
    "to_city"
).agg(count("*").alias("booking_count"))

win_spec = Window.orderBy(col("booking_count").desc())

destination_df.withColumn(
    "dense_rank",
    dense_rank().over(win_spec)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------------+----------+
|  to_city|booking_count|dense_rank|
+---------+-------------+----------+
|    Delhi|            5|         1|
|Hyderabad|            4|         2|
|   Mumbai|            3|         3|
|  Chennai|            2|         4|
|  Kolkata|            2|         4|
|Bangalore|            2|         4|
|     Pune|            1|         5|
|      Goa|            1|         5|
+---------+-------------+----------+



Part 7 : Delta Lake

In [0]:
bookings.write.format('delta').mode('overwrite') \
        .save('/Volumes/airline_databricks_7405614278006752/delta_file/bronze_files/bookings')

bookings.write.format('delta').mode('overwrite') \
        .saveAsTable('bookings_delta')

In [0]:
spark.sql("""
CREATE TABLE booking_master
USING DELTA
AS SELECT * FROM bookings_delta
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
day2_data = [
    ("B001", "F101", "Rahul Sharma", "Economy", 15500, "2025-06-02"),
    ("B005", "F103", "Priya Nair", "Business", 25000, "2025-06-02"),
    ("B012", "F105", "Arjun Kumar", "Economy", 19500, "2025-06-02"),
    ("B020", "F107", "Sneha Reddy", "Business", 25000, "2025-06-02"),
    ("B030", "F102", "Vikram Singh", "Economy", 15500, "2025-06-02"),

   ("B201", "F112", "Aakash Bhandari", "Economy", 10800, "2026-06-09"),
    ("B202", "F113", "Ritu Saxena",  "Business",25600, "2026-06-09"),
    ("B203", "F114", "Devansh Oberoi",  "Economy",9700, "2026-06-09"),
    ("B204", "F115", "Lavanya Chandra", "Premium Economy",16400, "2026-06-09"),
    ("B205", "F101", "Nikhil Bose", "Business",22900, "2026-06-09"),
    ("B206", "F103", "Shalini Rawat", "Economy",10100, "2026-06-09"),
    ("B207", "F105", "Aditya Tandon", "Premium Economy",15800, "2026-06-09"),
    ("B208", "F107", "Bhavna Shetty", "Economy", 11200, "2026-06-09"),
    ("B209", "F110", "Mohit Sengupta",  "Business",24800,"2026-06-09"),
    ("B210", "F102", "Sarika Bedi", "Premium Economy", 17100, "2026-06-09"),
]

day2_columns = ["booking_id", "flight_id", "passenger_name", "travel_class", "ticket_price", "booking_date"]
day2_df = spark.createDataFrame(day2_data, day2_columns)
display(day2_df)

booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date
B001,F101,Rahul Sharma,Economy,15500,2025-06-02
B005,F103,Priya Nair,Business,25000,2025-06-02
B012,F105,Arjun Kumar,Economy,19500,2025-06-02
B020,F107,Sneha Reddy,Business,25000,2025-06-02
B030,F102,Vikram Singh,Economy,15500,2025-06-02
B201,F112,Aakash Bhandari,Economy,10800,2026-06-09
B202,F113,Ritu Saxena,Business,25600,2026-06-09
B203,F114,Devansh Oberoi,Economy,9700,2026-06-09
B204,F115,Lavanya Chandra,Premium Economy,16400,2026-06-09
B205,F101,Nikhil Bose,Business,22900,2026-06-09


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "bookings_delta"
)

delta_table.alias("target").merge(
    day2_df.alias("source"),
    "target.booking_id = source.booking_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Part 9 : Time Travel

In [0]:
version0 = spark.read.format("delta").option('versionAsOf', 0).table('bookings_delta')
version1 = spark.read.format("delta").option('versionAsOf', 1).table('bookings_delta')
last_version = spark.read.format("delta").table('bookings_delta')

print("Row count before merge :", version0.count())
print("Row count after merge  :", last_version.count())
print("New rows inserted      :", last_version.count() - version0.count())

Row count before merge : 20
Row count after merge  : 35
New rows inserted      : 15


In [0]:
%sql 
optimize bookings_delta;

optimize bookings_delta
zorder by travel_class;

vacuum bookings_delta;

path
""


In [0]:
%sql
DESCRIBE HISTORY bookings_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-06-20T16:46:13.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,VACUUM END,Map(status -> COMPLETED),null,List(308310693067330),0075afc8-cade-4441-81cd-40324d965317,0620-154800-vg0dc0ps-v2n,4,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/18.2.x-photon-scala2.13
4,2026-06-20T16:46:12.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)",null,List(308310693067330),0075afc8-cade-4441-81cd-40324d965317,0620-154800-vg0dc0ps-v2n,3,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/18.2.x-photon-scala2.13
3,2026-06-20T16:46:04.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(308310693067330),444677ad-4259-494c-9a26-18b76c09e4c6,0620-154800-vg0dc0ps-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 9, numRemovedBytes -> 18890, p25FileSize -> 2806, numDeletionVectorsRemoved -> 0, minFileSize -> 2806, numAddedFiles -> 1, maxFileSize -> 2806, p75FileSize -> 2806, p50FileSize -> 2806, numAddedBytes -> 2806)",null,Databricks-Runtime/18.2.x-photon-scala2.13
2,2026-06-20T16:42:48.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(booking_id#14066 = booking_id#14096)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(308310693067330),a81c9a62-991a-47c7-9c89-d06505313ca2,0620-154800-vg0dc0ps-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 8, numTargetBytesAdded -> 16323, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3234, materializeSourceTimeMs -> 433, numTargetRowsInserted -> 15, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1162, numTargetRowsUpdated -> 0, numOutputRows -> 15, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 15, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1560)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-20T16:28:51.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(308310693067330),9c5ded14-5c65-4c76-a316-04fd9e444b00,0620-154800-vg0dc0ps-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2567, numDeletionVectorsRemoved -> 0, numOutputRows -> 20, numOutputBytes -> 2567)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-20T16:27:17.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(30831069306

In [0]:
bookings.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("complete_bookings")

In [0]:
bookings.write.format('delta').mode('overwrite') \
        .save('/Volumes/airline_databricks_7405614278006752/delta_file/bronze_files/bookings')

In [0]:
spark.sql("""
CREATE TABLE external_booing
USING DELTA
AS SELECT * FROM bookings
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.table("bookings").createOrReplaceTempView("temp_booking")

In [0]:
spark.table("bookings").createOrReplaceTempView("global_booking")